In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark=SparkSession.builder.appName("SmartHomeEnergyTracker").getOrCreate()

In [0]:
data=[
(1,1,"Air Conditioner","Living Room",5.6,240,"2026-05-01 08:00:00"),
(2,2,"Ceiling Fan","Bedroom",0.85,360,"2026-05-03 09:00:00"),
(3,3,"Refrigerator","Kitchen",2.3,1440,"2026-05-08 10:00:00"),
(4,4,"Smart TV","Living Room",1.2,180,"2026-05-12 19:00:00"),
(5,5,"Speaker System","Home Theater",1.95,210,"2026-05-12 20:00:00"),
(6,6,"Water Heater","Bathroom",7.5,300,"2026-05-18 06:00:00"),
(7,7,"Dining Light","Dining Hall",0.35,480,"2026-05-22 18:00:00"),
(8,8,"Washing Machine","Guest Room",4.8,95,"2026-05-25 17:00:00"),
(9,9,"Desktop Computer","Study Room",2.1,300,"2026-05-29 22:00:00")
]

columns=["log_id","device_id","device_name","room_name","energy_kwh","usage_duration_minutes","timestamp"]
df=spark.createDataFrame(data,columns)
display(df)


log_id,device_id,device_name,room_name,energy_kwh,usage_duration_minutes,timestamp
1,1,Air Conditioner,Living Room,5.6,240,2026-05-01 08:00:00
2,2,Ceiling Fan,Bedroom,0.85,360,2026-05-03 09:00:00
3,3,Refrigerator,Kitchen,2.3,1440,2026-05-08 10:00:00
4,4,Smart TV,Living Room,1.2,180,2026-05-12 19:00:00
5,5,Speaker System,Home Theater,1.95,210,2026-05-12 20:00:00
6,6,Water Heater,Bathroom,7.5,300,2026-05-18 06:00:00
7,7,Dining Light,Dining Hall,0.35,480,2026-05-22 18:00:00
8,8,Washing Machine,Guest Room,4.8,95,2026-05-25 17:00:00
9,9,Desktop Computer,Study Room,2.1,300,2026-05-29 22:00:00


In [0]:
daily_summary=df.groupBy(to_date("timestamp").alias("usage_date")).agg(round(sum("energy_kwh"),2).alias("daily_energy_usage"))
display(daily_summary)

usage_date,daily_energy_usage
2026-05-01,5.6
2026-05-03,0.85
2026-05-08,2.3
2026-05-12,3.15
2026-05-18,7.5
2026-05-22,0.35
2026-05-25,4.8
2026-05-29,2.1


In [0]:
weekly_summary=df.groupBy(weekofyear("timestamp").alias("week_number")).agg(round(sum("energy_kwh"),2).alias("weekly_energy_usage"))
display(weekly_summary)

week_number,weekly_energy_usage
18,6.45
19,2.3
20,3.15
21,7.85
22,6.9


In [0]:
high_usage_devices=df.filter(col("energy_kwh")>4)
display(high_usage_devices.select("device_name","energy_kwh"))

device_name,energy_kwh
Air Conditioner,5.6
Water Heater,7.5
Washing Machine,4.8


In [0]:
daily_summary.write.format("delta").mode("overwrite").saveAsTable("daily_summary")

In [0]:
weekly_summary.write.format("delta").mode("overwrite").saveAsTable("weekly_summary")